In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [2]:
df = pd.read_csv('Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv')

In [4]:
features = ['instruction', 'category']
X = df[features]
y = df['intent']

In [5]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y,
                                                      train_size=0.8,
                                                      random_state=0)

In [8]:
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer

In [9]:
tfidf_preprocessor = ColumnTransformer(
    transformers=[
        ('subject_tfidf', TfidfVectorizer(stop_words='english', max_features=500), 'category'),
        ('desc_tfidf', TfidfVectorizer(stop_words='english', max_features=1000), 'instruction')
    ],
    remainder='passthrough'
)

In [12]:
pipe = Pipeline(steps=[
    ('preprocessor', tfidf_preprocessor),
    ('model', RandomForestClassifier(n_estimators=500, oob_score=True, random_state=0))
])

pipe.fit(X_train, y_train)

preds = pipe.predict(X_valid)
print("OOB Score with TfidfVectorizer: " + str(pipe.named_steps['model'].oob_score_))

OOB Score with TfidfVectorizer: 0.9827882960413081


In [13]:
print(f"Accuracy Score: {accuracy_score(y_valid, preds)}")
print(classification_report(y_valid, preds))

Accuracy Score: 0.9813953488372092
                          precision    recall  f1-score   support

            cancel_order       1.00      0.98      0.99       186
            change_order       0.98      0.97      0.97       202
 change_shipping_address       0.98      1.00      0.99       189
  check_cancellation_fee       1.00      1.00      1.00       175
           check_invoice       0.84      0.87      0.86       202
   check_payment_methods       1.00      1.00      1.00       193
     check_refund_policy       1.00      0.97      0.99       190
               complaint       1.00      1.00      1.00       208
contact_customer_service       1.00      0.99      0.99       206
     contact_human_agent       0.99      1.00      0.99       200
          create_account       0.98      0.98      0.98       194
          delete_account       0.98      0.99      0.99       203
        delivery_options       0.99      1.00      1.00       191
         delivery_period       1.00     

In [14]:
import joblib
joblib.dump(pipe, "model.pkl")

['model.pkl']